# Lab 3: ระบบ RAG Part II
RAG หรือ Retrieval-Augmented Generation คือ เทคนิคการเพิ่มความสามารถให้ LLMs โดยการดึงข้อมูลจากแหล่งข้อมูลภายนอกมาใช้ประกอบการสร้างคำตอบ แทนที่จะพึ่งพาเพียงความรู้ที่โมเดลมีอยู่เดิมจากการเทรน เพื่อลดการ Hallucination และทำให้สามารถตอบคำถามเกี่ยวกับข้อมูลเฉพาะชุดนั้นๆ ได้อย่างถูกต้องมากขึ้น


## เป้าหมาย: 

พัฒนาสร้างระบบ RAG (Retrieval-Augmented Generation) เป็นระบบแนะนำกระทู้ Pantip ที่เกี่ยวข้อง โดยอ้างอิงข้อมูล [krathu-500
](https://github.com/Pittawat2542/krathu-500) 

#### Part I: Indexing 

_... ติดตามใน Lab3.1_

#### Part II: Generating 

ขั้นตอนนี้ คือ การแปลง **คำถาม** จากผู้ใช้ ค้นหาบริบทที่เกี่ยวข้อง และ ประมวลผลเป็นคำตอบที่เหมาะสม

1. **Retrieve** — ค้นหาชิ้นเอกสารที่ใกล้เคียงที่สุด
2. **Generate** — ประมวลผลคำถาม + เอกสารที่เกี่ยวข้อง เพื่อออกแบบคำตอบ

โดยใช้ `InMemoryVectorStore` และเปรียบเทียบ Retrieval 3 รูปแบบ

1. **Simple Retrieval** — Dense semantic search
2. **Hybrid Retrieval** — Dense retrieval + BM25 และรวมอันดับด้วย Reciprocal Rank Fusion (RRF)
3. **Retrieval + Reranking** — Dense retrieval แบบกว้าง แล้วให้ LLM จัดอันดับใหม่ก่อนตอบ


![RAG Pipeline](https://docs.nvidia.com/nemo-framework/user-guide/24.12/_images/rag_pipeline.png "RAG Pipeline")

credit: https://docs.nvidia.com/nemo-framework/user-guide/24.12/rag/ragoverview.html

## Set up — โหลด Vector Store จาก Lab 3.1

ในการพัฒนา application จริง มักจัดเก็บและค้นคืน vectors ด้วย Vector Database เช่น Qdrant หรือ Pinecone ซึ่งรองรับการจัดเก็บข้อมูลแบบถาวร การค้นหาข้อมูลจำนวนมาก และการทำงานร่วมกันระหว่างหลายระบบ

อย่างไรก็ตาม เพื่อให้ Lab ไม่ซับซ้อนเกินไป จึงขอใช้ `InMemoryVectorStore` ซึ่งจัดเก็บ vectors ไว้ในหน่วยความจำของ Python process

ในขั้นตอนนี้ เราจะโหลด chunks, metadata และ vectors ที่สร้างไว้ใน `Lab 3.1` จากไฟล์ แล้วนำข้อมูลดังกล่าวมาสร้าง InMemoryVectorStore ขึ้นใหม่ วิธีนี้ช่วยให้สามารถนำ vectors เดิมกลับมาใช้ได้ทันที โดยไม่ต้องเรียก Embedding Model เพื่อประมวลผลเอกสารทั้งหมดซ้ำอีกครั้ง

### Load Dense Vector Store

In [2]:
import pickle
from pathlib import Path

input_path = Path("./Examples/documents_dense_vectors.pkl")
with input_path.open("rb") as file:
    index_data = pickle.load(file)

chunks = index_data["chunks"]
chunk_ids = index_data["chunk_ids"]
chunk_vectors = index_data["chunk_vectors"]

MODEL_NAME = index_data["model_name"]
OUTPUT_DIMENSION = index_data["output_dimension"]

print(f"Loaded {len(chunk_vectors):,} vectors")
print(f"Model: {MODEL_NAME}")
print(f"Vector dimension: {OUTPUT_DIMENSION}")

Loaded 1,230 vectors
Model: gemini-embedding-2
Vector dimension: 768


In [3]:
from langchain_core.vectorstores import InMemoryVectorStore
import os
from dotenv import load_dotenv
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI

load_dotenv()
GEMINI_KEY = os.getenv("GEMINI_KEY")
EMBEDED_NAME = "gemini-embedding-2"
GENERATION_MODEL = "gemini-3.5-flash-lite"
OUTPUT_DIMENSION = 768

embedder = GoogleGenerativeAIEmbeddings(
    api_key=GEMINI_KEY,
    model=MODEL_NAME,
    task_type="SEMANTIC_SIMILARITY",
    output_dimensionality=OUTPUT_DIMENSION,
)

llm = ChatGoogleGenerativeAI(
    model=GENERATION_MODEL,
    thinking_level="minimal",
)


In [4]:
import time

vector_store = InMemoryVectorStore(embedding=embedder)

indexing_started = time.perf_counter()
for chunk, chunk_id, vector in zip(
    chunks,
    chunk_ids,
    chunk_vectors,
    strict=True,
):
    vector_store.store[str(chunk_id)] = {
        "id": str(chunk_id),
        "vector": vector,
        "text": chunk.page_content,
        "metadata": chunk.metadata,
    }

indexing_seconds = time.perf_counter() - indexing_started

print(f"Indexed {len(vector_store.store):,} chunks")
print(f"Indexing time: {indexing_seconds:.4f} seconds")

Indexed 1,230 chunks
Indexing time: 0.0005 seconds


In [5]:
ls krathu-500

README.md                example/                 posts.csv
baseline-model/          labeled/                 requirements.txt
chromedriver*            main.py                  small-dataset-generator/
comments.csv             post-processing/


In [6]:
import pandas as pd

posts_df = pd.read_csv("./krathu-500/post-processing/posts.csv", dtype={"id": "string"})
comments_df = pd.read_csv(
    "./krathu-500/post-processing/comments.csv",
    dtype={
        "comment_id": "string",
        "reply_to": "string",
        "text": "string",
    },
)

In [7]:
# จำกัดจำนวน Posts ที่สนใจ แค่ 100 ข้อมูลเท่านั้น
posts_df = posts_df.head(100)

In [8]:
posts_df.head()

,id,title,url,comment_count,vote_count,published_at
0,38597354,ถึงคนที่มีทุกอย่างอย่างที่ฝันไว้แล้ว..ว่าจริงห...,https://pantip.com/topic/38597354,126,3,2019-02-26 23:06:26+06:42
1,41041562,เด็กจบใหม่กับความเครียดจากครอบครัวและการหางาน,https://pantip.com/topic/41041562,211,0,2021-10-15 01:21:22+06:42
2,41040956,พนักงานออฟฟิศ สิ่งไหนที่เจ้านายทำแล้ว พนักงานร...,https://pantip.com/topic/41040956,275,0,2021-10-14 19:51:54+06:42
3,41031597,มีวิธีรับมืออย่างไรเมื่อเจอกับคนที่ทำงานเก่งมา...,https://pantip.com/topic/41031597,496,15,2021-10-10 09:05:21+06:42
4,30467953,คนขายประกันที่บอกว่าไปเที่ยวเมืองนอก ได้เงินเด...,https://pantip.com/topic/30467953,538,0,2013-05-10 09:59:20+06:42


In [9]:
comments_df.head()

,comment_id,text,collected_at,published_at,reply_to
0,38597354-0,ผมคิดถึงเรื่องนี้มาหลายครั้ง แต่ก็ไม่เชิงว่าค...,2021-10-19 16:42:41.984684,2019-02-26 23:06:26+06:42,<NA>
1,38597354-7817b4e7-6c30-414a-8145-c96864d38409,ซื้อความรักไม่ได้ ถึงเปย์ก็ได้แต่คนไม่จริงใจ,2021-10-19 16:42:42.034824,2019-02-26 23:09:18+06:42,38597354.0
2,38597354-314fe8b7-2ded-4b50-990b-21bded1edbf5,อยากได้ ไม่ได้ทุกข์ พอได้แล้วสุข สุขแล้วเบื่อ\...,2021-10-19 16:42:42.064554,2019-02-27 00:00:36+06:42,38597354.0
3,38597354-4f1ae8a3-3eac-46f3-91de-eca33be44e94,ความสุขมันก็คือการได้รับในสิ่งที่ตัวเองต้องการ...,2021-10-19 16:42:42.096689,2019-02-27 02:31:00+06:42,38597354.0
4,38597354-38e7901d-9402-445d-a235-99d2e772c641,ความสุขของแต่ละคนมันอยู่ที่นิยามของเจ้าตัวครับ,2021-10-19 16:42:42.125382,2019-02-27 04:34:28+06:42,38597354.0


In [10]:
comments_df["post_id"] = comments_df["comment_id"].str.extract(r"^(\d+)-", expand=False)

In [11]:
print(f"posts.csv: {len(posts_df):,} records")
print(f"comments.csv: {len(comments_df):,} records")

posts.csv: 100 records
comments.csv: 63,867 records


In [12]:
from langchain_core.documents import Document

post_lookup = posts_df.set_index("id").to_dict(orient="index")
source_documents: list[Document] = []

for post_id, comments_group in comments_df.groupby("post_id", sort=False):
    if pd.isna(post_id) or post_id not in post_lookup:
        continue

    post = post_lookup[post_id]
    content_parts: list[str] = []

    for row in comments_group.itertuples(index=False):
        text = row.text
        if pd.isna(text) or (text is None) or str(text).strip()=="":
            continue

        is_post = str(row.comment_id).endswith("-0")
        source_type = "post" if is_post else "comment"
        label = "เนื้อหากระทู้" if is_post else "ความคิดเห็น"

        content_parts.append(
            f"ประเภทข้อมูล: {label}\n"
            f"รหัสข้อมูล: {row.comment_id}\n"
            f"เผยแพร่เมื่อ: {row.published_at}\n"
            f"ตอบกลับ: {str(row.reply_to)}\n"
            f"เนื้อหา:\n{text}"
        )

    if not content_parts:
        continue

    page_content = (
        f"ชื่อกระทู้: {post['title']}\n"
        f"URL: {post['url']}\n\n"
        + "\n\n---\n\n".join(content_parts)
    )

    if len(page_content) > 150000:
        continue

    source_documents.append(
        Document(
            page_content=page_content,
            metadata={
                "document_id": str(post_id),
                "post_id": str(post_id),
                "post_title": str(post["title"]),
                "source_type": "post_with_comments",
                "url": str(post["url"]),
                "content_count": len(content_parts),
                "comment_count": sum(
                    not str(comment_id).endswith("-0")
                    for comment_id in comments_group["comment_id"]
                ),
            },
        )
    )

print(f"source documents: {len(source_documents):,}")

source documents: 95


In [13]:
print(source_documents[0].page_content[0:500]+"...")

ชื่อกระทู้: ถึงคนที่มีทุกอย่างอย่างที่ฝันไว้แล้ว..ว่าจริงหรือ ที่เงินมันซื้อความสุขได้
URL: https://pantip.com/topic/38597354

ประเภทข้อมูล: เนื้อหากระทู้
รหัสข้อมูล: 38597354-0
เผยแพร่เมื่อ: 2019-02-26 23:06:26+06:42
ตอบกลับ: <NA>
เนื้อหา:
ผมคิดถึงเรื่องนี้มาหลายครั้ง  แต่ก็ไม่เชิงว่าครุ่นคิดอยู่ตลอดเวลา
แต่อารมณ์มันแบบ  แวบขึ้นมาในหัวเป็นพักๆ   เวลาที่รู้สึกว่าตัวเองไม่มีความสุขมาก อย่างที่ควรจะเป็น

เลยอยากรู้ว่า  มีใครเป็นเหมือนกันบ้าง   ที่พอเราเดินมาถึงจุดๆหนึ่งในชีวิตในแบบที่เราอยากจะเป็น...


In [14]:
document_ids = [doc.metadata["document_id"] for doc in source_documents]

document_summary = pd.DataFrame(
    {
        "document_id": document_ids,
        "characters": [len(doc.page_content) for doc in source_documents],
    }
)
print(f"documents: {len(source_documents):,}")
document_summary.describe(include="all")


documents: 95


,document_id,characters
count,95,95.000000
unique,95,NaN
top,38597354,NaN
freq,1,NaN
mean,NaN,51001.800000
std,NaN,24445.657951
min,NaN,19189.000000
25%,NaN,31678.000000
50%,NaN,45361.000000
75%,NaN,63819.500000


### Load BM25 Vector Store

In [15]:
!uv pip install -q pythainlp rank-bm25

In [16]:
import unicodedata
from pythainlp.tokenize import word_tokenize
from pythainlp.util import normalize as pythainlp_th_normalize
from rank_bm25 import BM25Okapi
import numpy as np

from langchain_core.documents import Document

def tokenize_thai(text: str) -> list[str]:
    normalized = unicodedata.normalize("NFKC", text).lower()
    normalized_th = pythainlp_th_normalize(normalized)
    tokens = word_tokenize(normalized_th, engine="newmm", keep_whitespace=False)
    return [token.strip() for token in tokens if token.strip()]
    return tokens


class ThaiBM25Index:
    def __init__(self, documents: list[Document]):
        self.documents = list(documents)
        self.tokenized_corpus = []

        for document in tqdm(documents, desc="Tokenizing"):
            tokens = tokenize_thai(document.page_content)
            self.tokenized_corpus += [tokens]
            
        self.document_index = None

    def index(self):
        self.document_index = BM25Okapi(self.tokenized_corpus)
        
    def search(self, query: str, k: int = 4, *, include_zero_scores: bool = False):
        query_tokens = tokenize_thai(query)
        scores = self.document_index.get_scores(query_tokens)
        ranked_indices = np.argsort(scores)[::-1]

        results: list[tuple[Document, float]] = []
        for index in ranked_indices:
            score = float(scores[index])
            if score <= 0 and not include_zero_scores:
                continue
            results.append((self.documents[int(index)], score))
            if len(results) >= k:
                break
        return results

In [17]:
import pickle
from pathlib import Path

input_path = Path("./Examples/documents_sparse_vectors.pkl")
with input_path.open("rb") as file:
    index_data = pickle.load(file)


bm25_index = index_data["chunk_vectors"]

In [18]:
# test_query = "อยากรวยไวๆ ควรทำยังไงครับ?"
# dense_preview = vector_store.similarity_search_with_score(test_query, k=3)
# bm25_preview = bm25_index.search(test_query, k=3)

## Set up LangGraph State

In [19]:
from typing import Any, TypedDict

class RAGState(TypedDict, total=False):
    question: str
    top_k: int
    contexts: list[dict[str, Any]]
    answer: str
    latency_seconds: float

## Pipeline 1 — Simple Retrieval

```mermaid
flowchart LR
  Q["Question"] --> R["Dense retrieve top-k"]
  R --> G["Generate grounded answer"]
```



Pipeline นี้เป็นกระบวนการ RAG ขั้นพื้นฐาน โดยเริ่มจากรับคำถามของผู้ใช้ แล้วค้นหา chunks ที่มีความหมายใกล้เคียงกับคำถามมากที่สุดจำนวน top-k รายการ จาก  Dense Vector Store จากนั้นจึงส่งคำถามพร้อมเนื้อหาที่ค้นพบให้ LLM เพื่อสร้างคำตอบโดยอ้างอิงจากข้อมูลดังกล่าว

In [20]:
def dense_retrieve_node(state: RAGState) -> dict[str, Any]:
    top_k = state.get("top_k", TOP_K)
    results = vector_store.similarity_search_with_score(
        state["question"],
        k=top_k,
    )

    contexts = []
    for rank, (document, score) in enumerate(results, start=1):
        contexts.append({
            "chunk_id": document.metadata["chunk_id"],
            "text": document.page_content,
            "metadata": dict(document.metadata),
            "retrieval_method": "dense",
            "rank": rank,
            "score": score,
            
        })

    return {
        "contexts": contexts,
    }

In [21]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

ANSWER_SYSTEM_PROMPT = """
คุณเป็นผู้ช่วยตอบคำถามจาก knowledge corpus ภาษาไทย

กติกา:
1. ใช้เฉพาะข้อมูลใน CONTEXT เท่านั้น
2. ใส่ citation ด้วย chunk_id ในวงเล็บเหลี่ยม เช่น [38597354-0::chunk-001]
3. citation ต้องอยู่หลังข้อความที่หลักฐานนั้นรองรับ
4. หาก context ไม่เพียงพอ ให้ระบุว่าไม่พบข้อมูลเพียงพอใน corpus
5. แยกข้อเท็จจริงจากความคิดเห็นของผู้ร่วมสนทนาให้ชัดเจน
6. ตอบอย่างกระชับและไม่แต่งข้อมูลเพิ่ม
""".strip()


def generate_answer_node(state: RAGState) -> dict[str, Any]:
    contexts = state.get("contexts", [])
    if not contexts:
        return {
            "answer": "ไม่พบข้อมูลที่เกี่ยวข้องเพียงพอใน corpus",
        }

    blocks = []
    for context in contexts:
        chunk_id = context["chunk_id"]
        blocks.append(f"# [ID={chunk_id}]\n{context['text']}")
        
    rendered_context = "\n\n".join(blocks)

    prompt = (
        f"QUESTION:\n{state['question']}\n\n"
        f"CONTEXT:\n{rendered_context}"
    )

    response = llm.invoke(
        [
            SystemMessage(content=ANSWER_SYSTEM_PROMPT),
            HumanMessage(content=prompt),
        ]
    )
    return {
        "answer": response
    }

In [22]:
from langgraph.graph import END, START, StateGraph

simple_builder = StateGraph(RAGState)
simple_builder.add_node("retrieve", dense_retrieve_node)
simple_builder.add_node("generate", generate_answer_node)

simple_builder.add_edge(START, "retrieve")
simple_builder.add_edge("retrieve", "generate")
simple_builder.add_edge("generate", END)
simple_rag_graph = simple_builder.compile()


In [23]:
import time
query = "อยากรวยไวๆ ควรทำยังไงครับ?"

TOP_K = 3
started = time.perf_counter()
simple_result = simple_rag_graph.invoke({"question": query, "top_k": TOP_K})
latency = time.perf_counter() - started

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


In [24]:
import pandas as pd
print(f"Latency: {latency:.2f} seconds")
pd.DataFrame(simple_result["contexts"])[["rank", "chunk_id", "retrieval_method", "score"]]

Latency: 3.16 seconds


,rank,chunk_id,retrieval_method,score
0,1,35725482::chunk-019,dense,0.653792
1,2,32260495::chunk-008,dense,0.618318
2,3,41003042::chunk-008,dense,0.608228


In [25]:
from IPython.display import Markdown, display

Markdown(simple_result["answer"].content[0]["text"])


จากข้อมูลในคลังความรู้ ความคิดเห็นของผู้ร่วมสนทานาได้เสนอแนวทางและเครื่องมือที่เกี่ยวข้องกับการสร้างรายได้และการสะสมความมั่งคั่ง ดังนี้ครับ:

* **ทำธุรกิจหรือทำการค้า:** คนรวยจำนวนมากเริ่มต้นจากการทำสิ่งที่เข้าใจง่าย มีความต้องการสูง เช่น ทำธุรกิจร้านขายวัสดุอุปกรณ์แล้วขยายกิจการ และต้องรู้จักใช้เงินขยายกิจการต่อไปเรื่อยๆ [35725482::chunk-019]
* **มีรายรับหลายทางหรือเน้น Passive Income:** การมีรายรับจากหลายช่องทางและการโฟกัสเรื่อง Passive Income [35725482::chunk-021]
* **การลงทุนในรูปแบบต่างๆ:** เช่น เล่นหุ้น, ซื้อขายที่ดิน, เล่นบิทคับ (รวมถึงวิธี Dollar-cost averaging bitcoin) หรือซื้อกองทุน [35725482::chunk-019, 35725482::chunk-021, 41003042::chunk-010]
* **การหาความรู้:** ต้นทุนที่ต้องรีบหาก่อนเงินคือความรู้ และควรศึกษาให้ดีก่อนลงทุน [35725482::chunk-021, 41003042::chunk-010]
* **การออมและการมีวินัยทางการเงิน:** เช่น การเปิดบัญชีฝากประจำ เก็บเงินเป็นเปอร์เซ็นต์จากเงินเดือน ทำบัญชีรายรับ-รายจ่าย และฝึกวินัยทางการเงินควบคู่ไปด้วย [41003042::chunk-009, 41003042::chunk-010]

## Pipeline 2 — Hybrid Retrieval: Dense + BM25


Pipeline นี้ใช้ **Hybrid Retrieval** โดยค้นหาเอกสารผ่าน Retrieval สองรูปแบบที่มีจุดเด่นแตกต่างกัน แล้วนำผลลัพธ์มารวมกันก่อนส่งให้ LLM สร้างคำตอบ

* **Dense Retrieval** ค้นหาจากความใกล้เคียงทางความหมาย จึงสามารถค้นพบข้อความที่ใช้ถ้อยคำต่างจากคำถาม แต่มีความหมายสอดคล้องกัน
* **BM25 Retrieval** ค้นหาจากคำที่ปรากฏตรงกันระหว่างคำถามกับเอกสาร จึงเหมาะกับชื่อเฉพาะ ศัพท์เทคนิค รหัส หรือข้อความที่ต้องการความตรงกันของคำ

อย่างไรก็ตาม Dense cosine score และ BM25 score มีความหมายและช่วงค่าต่างกัน จึงไม่ควรนำคะแนนมาบวกกันโดยตรง Lab นี้ใช้ **Reciprocal Rank Fusion (RRF)** ซึ่งรวมผลลัพธ์โดยพิจารณาอันดับของเอกสารในแต่ละ ranked list แทนการเปรียบเทียบ raw score


$$
\operatorname{RRF}(d)=\sum_{r \in R}\frac{1}{k+\operatorname{rank}_r(d)}
$$

โดย $R$ คือชุดของ ranked lists และ $k$ เป็นค่าคงที่ที่ลดผลกระทบของอันดับต้นมากเกินไป ใน Lab ใช้ `RRF_K = 60`

เอกสารที่ปรากฏอยู่ในอันดับต้นของหลาย ranked lists จะได้รับคะแนน RRF สูงขึ้น ขณะที่เอกสารที่พบจาก Retrieval เพียงรูปแบบเดียวก็ยังมีโอกาสถูกเลือกเข้าสู่ผลลัพธ์สุดท้าย


```mermaid
flowchart TB
  Q["Question"] --> D["Dense candidates"]
  Q --> B["BM25 candidates"]
  D --> F["RRF fusion"]
  B --> F
  F --> G["Generate grounded answer"]
```

In [26]:
def reciprocal_rank_fusion(ranked_results, top_k, rrf_k = 60):
    fused = {}

    for method, results in ranked_results.items():
        for rank, (document, raw_score) in enumerate(results, start=1):
            chunk_id = document.metadata["chunk_id"]
            record = {
                "document": document,
                "rrf_score": 0.0,
                "dense_rank": None,
                "dense_score": None,
                "bm25_rank": None,
                "bm25_score": None,
            }
        
            if chunk_id in fused:
                record = fused[chunk_id]

            record["rrf_score"] += 1.0 / (rrf_k + rank)
            record[f"{method}_rank"] = rank
            record[f"{method}_score"] = float(raw_score)

            fused[chunk_id] = record

    return sorted(
        fused.values(),
        key=lambda item: item["rrf_score"],
        reverse=True,
    )[:top_k]

In [27]:
def hybrid_retrieve_node(state: RAGState) -> dict[str, Any]:
    top_k = state.get("top_k", TOP_K)
    candidate_k = state.get("candidate_k", top_k * 2)

    dense_results = vector_store.similarity_search_with_score(state["question"], k=candidate_k)
    bm25_results = bm25_index.search(state["question"], k=candidate_k)

    fused = reciprocal_rank_fusion(
        {"dense": dense_results, "bm25": bm25_results},
        top_k=top_k,
        rrf_k=60,
    )

    contexts = []
    for final_rank, item in enumerate(fused, start=1):
        document = item["document"]
        contexts.append({
            "chunk_id": document.metadata["chunk_id"],
            "text": document.page_content,
            "metadata": dict(document.metadata),
            "retrieval_method": "hybrid_rrf",
            "rank": final_rank,
            "score": float(item["rrf_score"]),

            "dense_rank": item["dense_rank"],
            "dense_score": item["dense_score"],
            "bm25_rank": item["bm25_rank"],
            "bm25_score": item["bm25_score"],
            
        })

    return {
        "contexts": contexts
    }


In [28]:
hybrid_builder = StateGraph(RAGState)
hybrid_builder.add_node("hybrid_retrieve", hybrid_retrieve_node)
hybrid_builder.add_node("generate", generate_answer_node)


hybrid_builder.add_edge(START, "hybrid_retrieve")
hybrid_builder.add_edge("hybrid_retrieve", "generate")
hybrid_builder.add_edge("generate", END)
hybrid_rag_graph = hybrid_builder.compile()


In [29]:
import time
query = "อยากรวยไวๆ ควรทำยังไงครับ?"

TOP_K = 3
started = time.perf_counter()
hybrid_result = hybrid_rag_graph.invoke({"question": query, "top_k": TOP_K})
latency = time.perf_counter() - started

In [30]:
import pandas as pd
print(f"Latency: {latency:.2f} seconds")
pd.DataFrame(hybrid_result["contexts"])[["rank", "chunk_id", "score", "dense_rank", "bm25_rank", "dense_score", "bm25_score"]]

Latency: 3.09 seconds


,rank,chunk_id,score,dense_rank,bm25_rank,dense_score,bm25_score
0,1,35725482::chunk-019,0.016393,1.0,NaN,0.653792,NaN
1,2,39945477::chunk-007,0.016393,NaN,1.0,NaN,18.975288
2,3,32260495::chunk-008,0.016129,2.0,NaN,0.618318,NaN


In [31]:
from IPython.display import Markdown, display

Markdown(hybrid_result["answer"].content[0]["text"])


จากข้อมูลใน context แนวทางที่เกี่ยวข้องกับการสร้างรายได้หรือความร่ำพึงมีดังนี้ครับ:

* **การทำธุรกิจหรือค้าขาย:** คนรวยจำนวนมากทำสิ่งที่เข้าใจง่าย มีความต้องการสูง ใช้เงินขยายกิจการต่อไปเรื่อยๆ (เช่น เริ่มจากร้านวัสดุอุปกรณ์แล้วขยายกิจการ) [35725482::chunk-019] และบางคนมีความตั้งใจอยากจะออกไปทำธุรกิจส่วนตัว [32260495::chunk-008]
* **การลงทุนและรายได้หลายทาง:** มีความเห็นแนะนำให้มีรายรับจากหลายทาง หรือโฟกัสเรื่อง passive income เช่น การลงทุนเล่นหุ้น, เล่นบิทคับ, ซื้อที่ดินขายต่อ [35725482::chunk-019], การลงทุนแบบ dollar-cost averaging bitcoin [35725482::chunk-019] รวมถึงการมีความรู้เป็นต้นทุนที่ต้องรีบหาก่อนเงิน [35725482::chunk-019]
* **ความคิดเห็นของผู้ร่วมสนทนา:** มีผู้ร่วมสนทนารายหนึ่งระบุความต้องการว่าอยากถูกหวยและอยากได้เงินเพื่อนำไปเรียนต่อกับการลงทุน [32260495::chunk-008]

## Pipeline 3 — Retrieval + LLM Reranking

Pipeline นี้เพิ่มขั้นตอน **LLM Reranking** เพื่อประเมินความเกี่ยวข้องของเอกสารอย่างละเอียดก่อนนำไปสร้างคำตอบ โดยแบ่งกระบวนการ Retrieval ออกเป็นสองช่วง:

1. **Candidate Retrieval** — ใช้ Dense Retrieval ค้นหาเอกสารเบื้องต้นจำนวน `candidate_k` รายการ ซึ่งมากกว่าจำนวนเอกสารที่ต้องการใช้จริง
2. **LLM Reranking** — ให้ LLM ประเมิน candidate แต่ละรายการเทียบกับคำถาม จัดลำดับใหม่อีกครั้ง และเลือกเอกสารที่เกี่ยวข้องที่สุดจำนวน `top_k` รายการ

```mermaid
flowchart LR
  Q["Question"] --> R["Retrieve candidate-k"]
  R --> RR["LLM rerank"]
  RR --> G["Generate from final top-k"]
```

In [32]:
from pydantic import BaseModel, Field

class RerankedItem(BaseModel):
    chunk_id: str = Field(description="An exact chunk_id from the candidates")
    relevance_score: float = Field(ge=0, le=100)
    reason: str = Field(description="Short reason grounded in the candidate text")


class RerankOutput(BaseModel):
    items: list[RerankedItem]

In [33]:
RERANK_SYSTEM_PROMPT = """
คุณเป็น retrieval reranker

ให้ประเมินว่า candidate แต่ละรายการช่วยตอบ QUESTION ได้โดยตรงเพียงใด
- ใช้เฉพาะ chunk_id ที่ให้มา
- ให้คะแนน 0-100
- คืนทุกรายการอย่างละหนึ่งครั้ง
- คะแนนสูงหมายถึงมีหลักฐานที่ตอบคำถามได้โดยตรงและเฉพาะเจาะจง
- อย่าให้คะแนนสูงเพียงเพราะมีคำบางคำตรงกัน แต่เนื้อหาไม่ตอบคำถาม
""".strip()

reranker = llm.with_structured_output(RerankOutput)

def rerank_node(state: RAGState):
    candidates = state.get("contexts", [])
    top_k = state.get("rerank_k", TOP_K)

    if not candidates:
        return {
            "contexts": []
        }

    texts = []
    for item in candidates:
        t = ""
        t += f"====== START CHUNK ======\n"
        t += f"## CHUNK_ID: {item['chunk_id']}\n"
        t += f"## TEXT:\n{item['text']}\n"
        t += f"====== END CHUNK ======\n"
        texts.append(t)
        
    candidate_text = "\n\n".join(texts)
    
    output = reranker.invoke([
        SystemMessage(content=RERANK_SYSTEM_PROMPT),
        HumanMessage(
            content=(
                f"# QUESTION:\n{state['question']}\n\n"
                f"# CANDIDATES:\n{candidate_text}"
            )
        ),
    ])

    candidate_by_id = {item["chunk_id"]: item for item in candidates}
    valid_scores = {}
    for item in output.items:
        if item.chunk_id in candidate_by_id:
            previous = valid_scores.get(item.chunk_id)
            if previous is None or item.relevance_score > previous.relevance_score:
                valid_scores[item.chunk_id] = item

    ranked_ids = sorted(
        valid_scores,
        key=lambda chunk_id: valid_scores[chunk_id].relevance_score,
        reverse=True,
    )

    # หาก model คืนมาไม่ครบ ให้ต่อท้าย candidate ที่ขาดตาม dense rank เดิม
    ranked_ids.extend(item["chunk_id"] for item in candidates if item["chunk_id"] not in valid_scores)

    reranked_contexts = []
    for final_rank, chunk_id in enumerate(ranked_ids[:top_k], start=1):
        original = candidate_by_id[chunk_id]
        score_item = valid_scores.get(chunk_id)
        reranked_contexts.append(
            {
                **original,
                "retrieval_method": "rerank",
                "rank": final_rank,
                "rerank_score": (float(score_item.relevance_score) if score_item else None),
                "rerank_reason": score_item.reason if score_item else "fallback",
            }
        )

    return {
        "contexts": reranked_contexts,
    }


In [34]:
rerank_builder = StateGraph(RAGState)
rerank_builder.add_node("retrieve_candidates", dense_retrieve_node)
rerank_builder.add_node("rerank", rerank_node)
rerank_builder.add_node("generate", generate_answer_node)


rerank_builder.add_edge(START, "retrieve_candidates")
rerank_builder.add_edge("retrieve_candidates", "rerank")
rerank_builder.add_edge("rerank", "generate")
rerank_builder.add_edge("generate", END)
rerank_rag_graph = rerank_builder.compile()


In [35]:
import time
query = "อยากรวยไวๆ ควรทำยังไงครับ?"

TOP_K = 3
started = time.perf_counter()
rerank_result = rerank_rag_graph.invoke({"question": query, "top_k": TOP_K*2, "rerank_k": TOP_K})
latency = time.perf_counter() - started

In [36]:
import pandas as pd
print(f"Latency: {latency:.2f} seconds")
pd.DataFrame(rerank_result["contexts"])[["rank", "chunk_id", "rank", "score", "rerank_score", "rerank_reason"]]

Latency: 5.52 seconds


,rank,chunk_id,rank,score,rerank_score,rerank_reason
0,1,35725482::chunk-019,1,0.653792,65.0,Discusses how people get rich through business...
1,2,41003042::chunk-008,2,0.608228,25.0,"Focuses on saving money, budgeting, and basic ..."
2,3,32260495::chunk-008,3,0.618318,10.0,Mentions starting a private business and brief...


In [37]:
from IPython.display import Markdown, display

Markdown(rerank_result["answer"].content[0]["text"])


จากข้อมูลในระบบ แนวทางที่เกี่ยวข้องกับการสร้างรายได้และการสะสมความมั่งคั่งจากความคิดเห็นของผู้ร่วมสนทนามีดังนี้ครับ:

* **ทำธุรกิจส่วนตัวหรือค้าขาย:** เริ่มต้นทำสิ่งที่เข้าใจง่าย มีความต้องการสูง และใช้เงินขยายกิจการต่อไปเรื่อยๆ เช่น การเปิดร้านขายวัสดุก่อสร้างและเครื่องใช้ไฟฟ้า [35725482::chunk-019]
* **สร้างรายได้หลายทางและเน้น Passive Income:** เช่น การลงทุนในรูปแบบ Dollar-Cost Averaging (DCA) บิทคอยน์ (Bitcoin) เล่นหุ้น ซื้อที่ดิน หรือลงทุนในคริปโต [35725482::chunk-019, 35725482::chunk-020, 41003042::chunk-011]
* **พัฒนาความรู้และวินัยทางการเงิน:** ต้องรีบหาความรู้ก่อนเรื่องเงิน และฝึกฝนวินัยทางการเงินควบคู่ไปกับการใช้เครื่องมือช่วยเก็บเงินหรือลงทุน เช่น การเปิดบัญชีฝากประจำ ทำประกันชีวิต หรือซื้อกองทุน [35725482::chunk-020, 41003042::chunk-011]
* นอกจากนี้ ยังมีความคิดเห็นของผู้ร่วมสนทนาที่พูดถึงความต้องการอยากถูกหวยเพื่อให้ได้เงินมาลงทุนและเรียนต่อ [32260495::chunk-010]

## สรุป

### * เมื่อใด Simple Retrieval เพียงพอ

- corpus มีขนาดเล็ก
- คำถามกับเอกสารใช้ถ้อยคำต่างกัน แต่มีความหมายใกล้เคียง
- latency และความเรียบง่ายสำคัญกว่าการเพิ่ม retrieval stage

### * เมื่อใด Hybrid Retrieval มีประโยชน์

- มีชื่อเฉพาะ รหัส ตัวเลข หรือศัพท์เทคนิคที่ต้อง match ตรงตัว
- ภาษาใน query และเอกสารอาจมีทั้งคำเดียวกันและคำพ้อง
- ต้องการเพิ่ม recall โดยไม่เพิ่ม LLM call

### * เมื่อใดควรเพิ่ม Reranking

- dense retriever หา candidate ที่เกี่ยวข้องได้ แต่ลำดับ top-k ยังไม่แม่น
- corpus มีเอกสารหน้าตาคล้ายกันจำนวนมาก
- ยอมรับ latency และค่าใช้จ่ายเพิ่มได้

## Evaluation

การประเมินระบบ RAG สามารถใช้กรอบ **RAG Triad** ซึ่งประกอบด้วยตัวชี้วัดหลัก 3 ด้าน ได้แก่

1. **Contextual Relevancy** ตรวจสอบว่า Context ที่ระบบค้นคืนมามีความเกี่ยวข้องกับคำถามหรือไม่
2. **Faithfulness หรือ Groundedness** ตรวจสอบว่าข้อกล่าวอ้างในคำตอบมีหลักฐานรองรับจาก Context และไม่มีข้อมูลที่ระบบแต่งขึ้นหรือไม่
3. **Answer Relevancy** ตรวจสอบว่าคำตอบตอบตรงกับคำถามและความต้องการของผู้ใช้หรือไม่

<img src="https://d2lsxfc3p6r9rv.cloudfront.net/rag-triad.svg" width="600">


ถ้ากำหนดให้

* \(Q\) = Question หรือคำถาม
* \(C\) = Retrieved Context หรือบริบทที่ระบบค้นคืนมา
* \(A\) = Answer หรือคำตอบที่ LLM สร้างขึ้น

| RAG Triad                   |             | คำถามที่ใช้ประเมิน                               | ส่วนของระบบที่เกี่ยวข้อง |
| --------------------------- | ----------: | ------------------------------------------------ | ------------------------ |
| Contextual Relevancy        | \(C \| Q\) | Context ที่ค้นคืนมาเกี่ยวข้องกับคำถามหรือไม่     | Retriever                |
| Faithfulness / Groundedness | \(A \| C\) | ข้อความในคำตอบมีหลักฐานรองรับจาก Context หรือไม่ | Generator                |
| Answer Relevancy            | \(A \| Q\) | คำตอบตอบตรงตามสิ่งที่ผู้ใช้ถามหรือไม่            | คุณภาพคำตอบโดยรวม        |


อย่างไรก็ตาม ยังมี metrics อื่นๆ ที่สามารถใช้ประเมินระบบ RAG ได้อีก เช่น

* Retrieval Precision@K
* Retrieval Recall@K
* Mean Reciprocal Rank หรือ MRR
* Mean Average Precision หรือ MAP
* Normalized Discounted Cumulative Gain หรือ NDCG
* ความถูกต้องและความครบถ้วนของ Citation
* คุณภาพและความครอบคลุมของ Corpus
* ระยะเวลาในการประมวลผล
* จำนวน Token และค่าใช้จ่าย
* ความทนทานต่อคำผิด การใช้คำพ้อง และการเรียบเรียงคำถามใหม่
* ความปลอดภัย ความเป็นส่วนตัว และการควบคุมสิทธิ์เข้าถึง
* ความถูกต้องเมื่อเทียบกับข้อเท็จจริงในโลกภายนอก